# Debugging Transaction Matching FailuresThis notebook inspects the reconciliation logic between the internal Rappi dataset and the third-party dataset. It surfaces issues around join keys, timestamp tolerances, and data cleanliness to explain why the initial reconciliation produced zero matches.

## 1. Environment SetupImport the required libraries and define helper utilities for loading the datasets from the expected locations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


In [ ]:
from src.utils import load_csv_with_fallback, build_timestamp

internal_raw = load_csv_with_fallback('base_interna.csv')
internal_raw['TIMESTAMP_RAPPI'] = build_timestamp(
    internal_raw,
    primary_col='CREATED_AT_RAPPI',
    date_col='FECHA_RAPPI',
    time_col='HORA_RAPPI',
)

third_party_raw = load_csv_with_fallback('base_tercero.csv')
third_party_raw['TIMESTAMP_TERCERO'] = build_timestamp(
    third_party_raw,
    primary_col='TRANSACTION_DATETIME_TERCERO',
    date_col='FECHA_TERCERO',
    time_col='HORA_TERCERO',
)


In [ ]:
import yaml
from pprint import pprint
from pathlib import Path

config_path = Path('config.yml')
with config_path.open('r', encoding='utf-8') as fh:
    config = yaml.safe_load(fh)
pprint(config)


## 2. Load Data and Derive TimestampsParse the raw CSV files, standardize the timestamp columns, and prepare normalized join keys for downstream diagnostics.

In [ ]:
# Quick preview confirms timestamps were parsed as expected.
internal_raw[['TIMESTAMP_RAPPI']].head(), third_party_raw[['TIMESTAMP_TERCERO']].head()



In [ ]:
third_party_raw[['TIMESTAMP_TERCERO', 'TRANSACTION_DATETIME_TERCERO', 'FECHA_HORA_TERCERO']].head()

## 3. Filter the Movement Types Involved in ReconciliationFocus on the movement types that should be matching between the two datasets before computing diagnostics.

In [ ]:
# Movement filters derived from the reconciliation specification.internal_focus = internal_raw[    internal_raw['MOVIMIENTO_RAPPI'].str.lower().isin(['disperse', 'debit'])].copy()third_focus = third_party_raw[    third_party_raw['DESCRIPCION_TERCERO'].str.lower().isin([        'transferencia ws', 'retiro en ventanilla ws', 'compra en pos'    ])].copy()print('Internal focus shape:', internal_focus.shape)print('Third-party focus shape:', third_focus.shape)

## 4. Normalize Join KeysStrip whitespace, harmonize casing, and pad the BIN/LAST_4 fields so we can analyze whether formatting mismatches are preventing joins.

In [ ]:
def normalize_string(series):    return series.fillna('').astype(str).str.strip().str.upper()# Prepare key components for internal data.internal_focus['IDENTIFICADOR_NORM'] = normalize_string(internal_focus['IDENTIFICADOR_RAPPI'])internal_focus['AUTH_CODE_NORM'] = normalize_string(internal_focus['AUTH_CODE_RAPPI'])internal_focus['BIN_NORM'] = normalize_string(internal_focus['BIN_NUMBER_RAPPI']).str.zfill(6)internal_focus['LAST4_NORM'] = normalize_string(internal_focus['FOUR_DIGITS_RAPPI']).str.zfill(4)internal_focus['TIMESTAMP_RAPPI'] = pd.to_datetime(internal_focus['TIMESTAMP_RAPPI'], errors='coerce')# Prepare key components for third-party data.third_focus['IDENTIFICADOR_NORM'] = normalize_string(third_focus['IDENTIFICADOR_TERCERO'])third_focus['AUTH_CODE_NORM'] = normalize_string(third_focus['AUTH_CODE_TERCERO'])third_focus['BIN_NORM'] = normalize_string(third_focus['BIN_TERCERO']).str.zfill(6)third_focus['LAST4_NORM'] = normalize_string(third_focus['LAST_FOUR_DIGITS_TERCERO']).str.zfill(4)third_focus['TIMESTAMP_TERCERO'] = pd.to_datetime(third_focus['TIMESTAMP_TERCERO'], errors='coerce')key_cols = ['IDENTIFICADOR_NORM', 'AUTH_CODE_NORM', 'BIN_NORM', 'LAST4_NORM']internal_focus[key_cols + ['TIMESTAMP_RAPPI']].head()

In [ ]:
third_focus[key_cols + ['TIMESTAMP_TERCERO']].head()

## 5. Audit Join-Key CompletenessQuantify nulls, empties, and unique counts for each join column to identify gaps that would prevent matches.

In [ ]:
def key_quality_report(df, prefix):    summary = []    for col in key_cols:        raw_col = col.replace('_NORM', f'_{prefix}')        series = df[col]        summary.append({            'column': raw_col,            'null_count': series.isna().sum(),            'empty_or_blank': (series == '').sum(),            'unique_non_null': series[series.notna() & (series != '')].nunique(),        })    return pd.DataFrame(summary)internal_key_report = key_quality_report(internal_focus, 'RAPPI')third_key_report = key_quality_report(third_focus, 'TERCERO')internal_key_report, third_key_report

### Unique Counts ComparisonCompare unique cardinalities between the internal and third-party AUTH codes and other key fields.

In [ ]:
unique_comparison = pd.DataFrame({    'metric': ['rows', 'auth_code_unique', 'identifier_unique', 'bin_unique', 'last4_unique'],    'internal': [        internal_focus.shape[0],        internal_focus['AUTH_CODE_NORM'].nunique(),        internal_focus['IDENTIFICADOR_NORM'].nunique(),        internal_focus['BIN_NORM'].nunique(),        internal_focus['LAST4_NORM'].nunique(),    ],    'third_party': [        third_focus.shape[0],        third_focus['AUTH_CODE_NORM'].nunique(),        third_focus['IDENTIFICADOR_NORM'].nunique(),        third_focus['BIN_NORM'].nunique(),        third_focus['LAST4_NORM'].nunique(),    ]})unique_comparison

## 6. Timestamp Alignment DiagnosticsFor records sharing the same normalized join keys, compute the distribution of absolute time deltas to verify whether the ±1 hour tolerance is appropriate.

In [ ]:
# Build a candidate pool by joining on all normalized keys (no time filter yet).
candidate_pairs = internal_focus.dropna(subset=key_cols + ['TIMESTAMP_RAPPI']).merge(
    third_focus.dropna(subset=key_cols + ['TIMESTAMP_TERCERO']),
    on=key_cols,
    suffixes=('_INT', '_THIRD'),
)
if not candidate_pairs.empty:
    candidate_pairs['time_diff_seconds'] = (
        candidate_pairs['TIMESTAMP_RAPPI'] - candidate_pairs['TIMESTAMP_TERCERO']
    ).dt.total_seconds().abs()
    time_stats = candidate_pairs['time_diff_seconds'].describe(percentiles=[0.5, 0.75, 0.9, 0.95])
    time_stats
else:
    print('No candidate pairs found when joining on normalized keys.')


In [ ]:
if not candidate_pairs.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(candidate_pairs['time_diff_seconds'] / 60, bins=50, color='C0', edgecolor='black')
    ax.set_title('Distribution of Time Differences (minutes) for Candidate Matches')
    ax.set_xlabel('Absolute time difference (minutes)')
    ax.set_ylabel('Count')
    plt.tight_layout()
else:
    print('Skipping histogram because no candidate pairs exist.')


### Inspect Large Time DifferencesShow a sample of potential matches with the largest time gaps to understand whether timezone misalignment or batch processing delays are present.

In [ ]:
if not candidate_pairs.empty:    display_columns = [        'TIMESTAMP_RAPPI', 'TIMESTAMP_TERCERO', 'time_diff_seconds',        'MOVIMIENTO_RAPPI', 'DESCRIPCION_TERCERO', 'AUTH_CODE_NORM'    ]    sample_pairs = (        candidate_pairs.assign(time_diff_minutes=candidate_pairs['time_diff_seconds'] / 60)        .sort_values('time_diff_seconds', ascending=False)        .head(20)    )    sample_pairs[display_columns + ['time_diff_minutes']]else:    print('No candidate pairs to inspect.')

## 7. Frequency Analysis of Key FieldsIdentify the most common AUTH codes and assess whether duplicates or repeated values might be skewing the join.

In [ ]:
internal_top_auth = (    internal_focus['AUTH_CODE_NORM']    .value_counts(dropna=False)    .head(10)    .rename_axis('AUTH_CODE')    .reset_index(name='internal_count'))third_top_auth = (    third_focus['AUTH_CODE_NORM']    .value_counts(dropna=False)    .head(10)    .rename_axis('AUTH_CODE')    .reset_index(name='third_party_count'))internal_top_auth, third_top_auth

## 8. Transaction Time DistributionsVisualize the timestamp distributions for each dataset to detect timezone shifts or missing intervals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)sns.histplot(    internal_focus['TIMESTAMP_RAPPI'].dropna(),    bins=50,    ax=axes[0])axes[0].set_title('Internal Disperse/Debit Timestamp Distribution')axes[0].set_xlabel('Timestamp')sns.histplot(    third_focus['TIMESTAMP_TERCERO'].dropna(),    bins=50,    ax=axes[1])axes[1].set_title('Third-Party WS/POS Timestamp Distribution')axes[1].set_xlabel('Timestamp')plt.tight_layout()

## 9. Crosstab of Movement Type CombinationsCreate a contingency table showing how often each internal movement type shares join keys with specific third-party descriptions (ignoring timestamp mismatches) to identify systematic misalignments.

In [ ]:
if not candidate_pairs.empty:    movement_crosstab = pd.crosstab(        candidate_pairs['MOVIMIENTO_RAPPI'],        candidate_pairs['DESCRIPCION_TERCERO']    )    movement_crosstabelse:    print('Unable to compute movement crosstab because no join-key overlaps were found.')

## 10. Suggested Remediation StepsSummarize potential corrective actions based on the diagnostics above.

- **Relax timestamp tolerance**: If the `time_diff_seconds` distribution shows many candidate pairs slightly above 3600 seconds, test a ±2 hour window or align timestamps to a common timezone.- **Normalize card identifiers**: Ensure both datasets left-pad BIN to six digits and last-four digits to four digits before matching.- **Handle missing or blank auth codes**: Records lacking `AUTH_CODE` will never match; consider fallback joins using transaction IDs or combine with `IDENTIFICADOR` + `ORDER_ID` where available.- **Investigate timezone offsets**: If histograms suggest a constant offset (e.g., 5 hours), adjust one dataset prior to reconciliation.- **Flag duplicated AUTH codes**: Cross-check repeated AUTH codes to verify whether they belong to multiple movements that require additional disambiguation.

In [ ]:
if not candidate_pairs.empty:
    movement_crosstab = pd.crosstab(
        candidate_pairs['MOVIMIENTO_RAPPI'],
        candidate_pairs['DESCRIPCION_TERCERO']
    )
    movement_crosstab
else:
    print('No candidate pairs available for movement crosstab.')
